In [2]:
import librosa
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf

In [4]:
import pandas as pd

# --------------------------------------------------
# File paths
# --------------------------------------------------

main_file = "xc_non_Eurasian.csv"



# --------------------------------------------------
# Load files
# --------------------------------------------------

df = pd.read_csv(main_file)

        

In [5]:
# Basic dataframe analysis

# Summary statistics for every column
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(dropna=False),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

display(summary)

# Number of duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

# Unique-value counts for each column
for column in df.columns:
    print(f"\n{'=' * 60}\n{column}")
    print(f"Unique values: {df[column].nunique(dropna=False)}")
    display(df[column].value_counts(dropna=False).head(20))

# Date analysis
dates = pd.to_datetime(df["date"], errors="coerce")
print("\nDate range:")
print(f"Earliest: {dates.min()}")
print(f"Latest:   {dates.max()}")

# Duration analysis
duration_seconds = pd.to_timedelta(
    "00:" + df["duration"].astype(str),
    errors="coerce"
).dt.total_seconds()

print("\nDuration statistics in seconds:")
display(duration_seconds.describe())

# Numeric summary
print("\nNumeric columns:")
display(df.describe(include="number"))

# Categorical summary
print("\nCategorical columns:")
display(df.describe(include="object").T)

,dtype,non_null,missing,unique_values,missing_percent
recording_id,int64,3473,0,3473,0.00
mic,object,2460,1013,323,29.17
species,object,3473,0,68,0.00
country,object,3473,0,67,0.00
date,object,3473,0,767,0.00
duration,object,3473,0,28,0.00
user,object,3473,0,446,0.00
recording_url,object,3473,0,3473,0.00


Duplicate rows: 0

recording_id
Unique values: 3473


1135246    1
1020912    1
1020968    1
1020965    1
1020953    1
1020952    1
1020950    1
1020948    1
1020947    1
1020946    1
1020945    1
1020918    1
1020917    1
1020910    1
1020943    1
1020907    1
1020906    1
1020905    1
1020903    1
1007580    1
Name: recording_id, dtype: int64


mic
Unique values: 323


NaN                                   1013
Primo EM 272                           530
Primo EM272                            115
Sennheiser MKH 416                     105
Dodotronic Hi-Sound DR                 104
Sm4                                     94
Tascam                                  43
Audiomoth                               43
Smmicra                                 37
Telinga pro 9 mkII                      35
telinga pro 9 mkII                      35
External USB                            34
Stereo EM272 capsules                   34
Telinga pro 9 MKII                      31
Tascam.                                 29
2 Primo EM272Z1                         24
Audiomoth 1.2.0                         24
AOM-5024L                               23
SM4                                     22
Dpa 4060s in parabolic reflector …      20
Name: mic, dtype: int64


species
Unique values: 68


... (13 outputs left out)


Numeric columns:


,recording_id
count,3.473000e+03
mean,1.021177e+06
std,8.640146e+04
min,8.590590e+05
25%,9.408640e+05
50%,1.021189e+06
75%,1.081102e+06
max,1.172421e+06



Categorical columns:


,count,unique,top,freq
mic,2460,322,Primo EM 272,530
species,3473,68,Common Snipe,347
country,3473,67,France,819
date,3473,767,2026-05-29,76
duration,3473,28,0:08,285
user,3473,446,Stanislas Wroza,764
recording_url,3473,3473,https://xeno-canto.org/1135246,1


Downsample User Wroza

In [7]:
# Limit Stanislas Wroza's samples to 250 while keeping all other records
wroza_mask = df["user"].str.contains("Wroza", case=False, na=False)
wroza_samples = df[wroza_mask].sample(n=min(250, wroza_mask.sum()), random_state=42)
other_samples = df[~wroza_mask]

df_downsampled = pd.concat([other_samples, wroza_samples], ignore_index=True)

output_file = "xc_non_Eurasian_downsampled.csv"
df_downsampled.to_csv(output_file, index=False)

print(f"Saved {len(df_downsampled)} rows to {output_file}")
print(f"Wroza samples retained: {len(wroza_samples)}")

Saved 2959 rows to xc_non_Eurasian_downsampled.csv
Wroza samples retained: 250


In [9]:
# Count non-missing values in the "mic" column
print(df_downsampled["mic"].value_counts())

Primo EM 272                               182
Sennheiser MKH 416                         105
Sm4                                         94
Tascam                                      43
Audiomoth                                   38
                                          ... 
Tellinga parabolic dish with stereo MK3      1
unknown clippy                               1
Internal mics                                1
Parabol A                                    1
Inbyggd.                                     1
Name: mic, Length: 322, dtype: int64
